In [55]:
# init
from pathlib import Path
from importlib import reload
import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsPandas.helpers as tph
import toolsSync.main as tsm
import os
import boto3
from dotenv import load_dotenv
import pandas as pd
import copy

def pckgs_reload():
    reload(tgm)
    reload(tph)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
logger = tgl.initiate_logger('logger')
process_state = tgm.load(DATA_DIR / 'process_state.json')
print(len(process_state))

214


In [56]:
load_dotenv()
bucket_name = os.environ["B2_BUCKET_NAME"]
session = boto3.session.Session()
s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

# read bucket

In [65]:
task_meta = {
    'scrape':{'dir':Path("data/raw/osm countries queries/")}, 
    'clean':{'dir':Path("data/cleaned/")}, 
    'test_basic':{'dir':Path("data/tests results/osm basic test")},
    'test_first_level':{'dir':Path("data/tests results/osm first level test")},
    'test_duplicates':{'dir':Path("data/tests results/osm duplicates test")}
}
task_responses = {t:s3.list_objects_v2(Bucket=bucket_name, Prefix=task_meta[t]['dir'].as_posix()) for t in task_meta.keys()}
[len(res['Contents']) for task,res in task_responses.items()]

[18, 83, 2, 12, 14]

In [83]:
sync_state = copy.deepcopy(task_meta)
for task, res in task_responses.items():
    sync_state[task]['b2_countries'] = {Path(obj['Key']).parent.name for obj in res['Contents']}
    sync_state[task]['local_countries'] = {dir.name for dir in (ROOT / sync_state[task]['dir']).glob('*') if dir.is_dir()}
    sync_state[task]['b2_files'] = {obj['Key'] for obj in res['Contents']}
    sync_state[task]['local_files'] = {dir.relative_to(ROOT).as_posix() for dir in (ROOT / sync_state[task]['dir']).rglob('*') if dir.is_file()}
    b2_countries_not_in_local = tgm.complement(
        sync_state[task]['b2_countries'],
        sync_state[task]['local_countries']
    )
    b2_files_not_in_local = tgm.complement(sync_state[task]['b2_files'], sync_state[task]['local_files'])
    sync_state[task]['b2_files_not_in_local'] = b2_files_not_in_local
    
# get len
sync_state_resume = copy.deepcopy(sync_state)
for task,val in sync_state_resume.items():
    for k,v in sync_state_resume[task].items():
        if k in ['dir']: continue
        sync_state_resume[task][k] = len(v)

In [84]:
pd.DataFrame(sync_state_resume).T

,dir,b2_countries,local_countries,b2_files,local_files,b2_files_not_in_local
scrape,data\raw\osm countries queries,15,83,18,304,0
clean,data\cleaned,83,83,83,84,0
test_basic,data\tests results\osm basic test,2,83,2,85,0
test_first_level,data\tests results\osm first level test,6,65,12,70,5
test_duplicates,data\tests results\osm duplicates test,2,32,14,46,0


# download data

In [92]:
task = 'test_first_level'
to_download_countries = {Path(file).parent.name for file in sync_state[task]['b2_files_not_in_local']}
print(len(to_download_countries))
print(to_download_countries)

3
{'BosniaAndHerzegovina', 'Benin', 'Armenia'}


In [96]:
# In oder to sync them more easily, just delete the local directory
for country in to_download_countries:
    try:
        os.rmdir(task_meta[task]['dir'] / country)
    except:
        print(f"Directory for {country} doesn't exist")

Directory for BosniaAndHerzegovina doesn't exist
Directory for Benin doesn't exist
Directory for Armenia doesn't exist


In [97]:
tsm.donwload_country_data_from_bucket(to_download_countries, bucket_name, task_meta[task]['dir'], ROOT / task_meta[task]['dir'], s3, logger)

[2025-12-25 18:29:25] [INFO] :   * Countries to get data: 3
[2025-12-25 18:29:25] [INFO] :     * Found in B2: 3, Missing in B2: 0
[2025-12-25 18:29:25] [INFO] :   * Downloading data for countries: 3
[2025-12-25 18:29:25] [INFO] :     * Directory: 'data\tests results\osm first level test' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\tests results\osm first level test'
[2025-12-25 18:29:25] [INFO] :     * (1/3) Country Benin files found: 3
[2025-12-25 18:29:25] [INFO] :       * Skip download of existing file Benin_first_level_test_res_1.pkl
[2025-12-25 18:29:25] [INFO] :       * Skip download of existing file Benin_first_level_test_res_2.pkl
[2025-12-25 18:29:25] [INFO] :       * Skip download of existing file Benin_first_level_test_res_3.pkl
[2025-12-25 18:29:25] [INFO] :     * (2/3) Country BosniaAndHerzegovina files found: 1
[2025-12-25 18:29:25] [INFO] :       * Skip download of existing file BosniaAndHerzegovina_first_level_test_res_1.pkl
[

['Benin', 'BosniaAndHerzegovina', 'Armenia']

# upload data

In [21]:
dirs = [DATA_DIR / 'cleaned' / country for country in sync_state['clean']['local_not_in_b2']]
print(len(dirs))

81


In [27]:
config = {'root':ROOT, 's3':s3, 'logger':logger}
for dir in dirs:
    tsm.upload_dir_files_to_backblaze(dir, config)

[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Afghanistan
[INFO] : Number of files found: 1
[INFO] : Uploaded Afghanistan_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Albania
[INFO] : Number of files found: 1
[INFO] : Uploaded Albania_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Algeria
[INFO] : Number of files found: 1
[INFO] : Uploaded Algeria_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned\Andorra
[INFO] : Number of files found: 1
[INFO] : Uploaded Andorra_cleaned.pkl to Backblaze successfully
[INFO] : Uploading directory: c:\Users\gonta\D\study\full stack\projects\adminis